In [365]:
import pandas as pd
import numpy as np

train_data = pd.read_csv('Data/model_train.csv')
valid_data = pd.read_csv('Data/model_valid.csv')
test_data = pd.read_csv('Data/model_test.csv')

'UPB_Diff_0', 'UPB_Diff_1', 'UPB_Diff_2', 
         'UPB_Diff_3', 'UPB_Diff_4', 'UPB_Diff_5', 'UPB_Diff_6', 'UPB_Diff_7', 'UPB_Diff_8', 'UPB_Diff_9', 'UPB_Diff_10', 'UPB_Diff_11', 
         'UPB_Diff_12'

'NumberOfUnits', 'Channel', 'PropertyState'

In [366]:
bool_col = ['FirstTimeHomebuyerFlag', 'SuperConformingFlag', 'CreditScore_MissFLag', 'OriginalDTI_MissFLag', 'EstimatedLTV_all_MissFlag']

cat_col = ['OccupancyStatus', 'PropertyType', 'LoanPurpose', 'ProgramIndicator',
           'PropertyValMethod', 'BalloonIndicator']

num_col = ['CreditScore', 'MI_Pct', 'OriginalDTI', 'OriginalLTV', 'OriginalInterestRate',
         'OriginalLoanTerm', 'NumberOfBorrowers', 'DebtServiceRatio', 'LTV_DTI_Interaction', 'CreditScore_LTV_Ratio', 
         'CreditScore_DTI_Ratio', 'CompositeRiskScore', 'corr_UPB_LTV', 'UPB_mean', 'UPB_slope', 
         'EstimatedLTV_mean', 'EstimatedLTV_std', 'EstimatedLTV_slope', 'avg_repayment_ratio',
         'pct_months_late', 'repayment_trend_slope', 'EstimatedLTV_RollingStd3m', 
         'EstimatedLTV_ChangeVolatility', 'SellerName_RiskEnc', 'ServicerName_RiskEnc', 'UPB_RollingStd3m', 'EstimatedLTV_delta', 
         'OriginalUPB', 'UPB_std', 'UPB_delta', 'UPB_MaxChange', 'UPB_ChangeVolatility', 'UPB_RollingMeanDiff3m', 
         'EstimatedLTV_RollingMeanDiff3m', 'EstimatedLTV_MaxChange', 'std_repayment_ratio']

In [367]:
def test_result(df: pd.DataFrame, pipeline):
    scores = pipeline["clf"].score_samples(pipeline["prep"].transform(df))
    raw_anom = -scores
    
    min_v, max_v = np.min(raw_anom), np.max(raw_anom)
    anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

    submission = pd.DataFrame()

    submission['Id'] = df['Id']
    submission['target'] = anom_score

    return submission

In [368]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

In [369]:
X_train = train_data[cat_col + num_col + bool_col]
X_valid = valid_data[cat_col + num_col + bool_col]
X_test = test_data.drop(columns="Id")

y_valid = valid_data[['index', 'target']]

In [370]:
out = ['index', 'target']
y_valid = valid_data[out]

Isolation Forest

In [311]:
from sklearn.ensemble import IsolationForest

preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_col),
            ("num", StandardScaler(with_mean=False), num_col),
            ("bool", "passthrough", bool_col),
        ],
        sparse_threshold=1.0,
    )

model = IsolationForest(
    n_estimators=500,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline([("prep", preprocess), ("clf", model)])
pipe.fit(X_train)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [312]:
scores = pipe["clf"].score_samples(pipe["prep"].transform(X_valid))
raw_anom = -scores

y_valid["anom_score"] = raw_anom

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

ap = average_precision_score(y_valid["target"], y_valid["anom_score"])
roc_auc = roc_auc_score(y_valid["target"], y_valid["anom_score"])

print(f"AP: {ap:.4f}")
print(f"AUC-ROC : {roc_auc:.4f}")

AP: 0.1315
AUC-ROC : 0.5392


/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_99129/2893069197.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_valid["anom_score"] = raw_anom


HBOS

In [313]:
from pyod.models.hbos import HBOS

preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_col),
            ("num", StandardScaler(with_mean=False), num_col),
            ("bool", "passthrough", bool_col),
        ],
        sparse_threshold=1.0,
    )

# HBOS Model
hbos_model = HBOS(
    contamination=0.05,  # proportion of expected outliers
    n_bins=10,           # number of bins for histogram
    alpha=0.1            # regularization to avoid overfitting
)

# Pipeline
pipe = Pipeline([("prep", preprocess), ("clf", hbos_model)])
# Fit on training set
pipe.fit(X_train)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [314]:
scores = pipe["clf"].decision_function(pipe["prep"].transform(X_valid))
raw_anom = -scores

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

y_valid["anom_score"] = anom_score

ap = average_precision_score(y_valid["target"], y_valid["anom_score"])
roc_auc = roc_auc_score(y_valid["target"], y_valid["anom_score"])

print(f"AP: {ap:.4f}")
print(f"AUC-ROC : {roc_auc:.4f}")

AP: 0.1086
AUC-ROC : 0.4541


/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_99129/2162249464.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_valid["anom_score"] = anom_score


LOF

In [371]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, average_precision_score

y_val = y_valid['target']

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_col),
        ("num", StandardScaler(with_mean=False), num_col),
        ("bool", "passthrough", bool_col),
    ],
    sparse_threshold=1.0,
)

candidate_n_neighbors = [10, 20, 30, 40]
candidate_contamination = [0.01, 0.05, 0.1, 'auto']

X_train_preprocessed = preprocess.fit_transform(X_train)

best_params = None
best_ap = -np.inf
best_model = None

for n_neighbors in candidate_n_neighbors:
    for contamination in candidate_contamination:
        lof_model = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination, novelty=True)
        lof_model.fit(X_train_preprocessed)  # Unsupervised fit on training data

        # Transform validation data
        X_val_preprocessed = preprocess.transform(X_valid)

        # Get prediction scores on validation (negative outlier factor)
        scores = -lof_model.decision_function(X_val_preprocessed)

        # Calculate AP score (needs labeled y_val)
        ap = average_precision_score(y_val, scores)

        print(f"n_neighbors={n_neighbors}, contamination={contamination}, AP={ap:.4f}")

        if ap > best_ap:
            best_ap = ap
            best_params = {'n_neighbors': n_neighbors, 'contamination': contamination}
            best_model = lof_model

print("Best params:", best_params)
print("Best AP on validation:", best_ap)

n_neighbors=10, contamination=0.01, AP=0.2649
n_neighbors=10, contamination=0.05, AP=0.2649
n_neighbors=10, contamination=0.1, AP=0.2649
n_neighbors=10, contamination=auto, AP=0.2649
n_neighbors=20, contamination=0.01, AP=0.2634
n_neighbors=20, contamination=0.05, AP=0.2634
n_neighbors=20, contamination=0.1, AP=0.2634


KeyboardInterrupt: 

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_col),
        ("num", StandardScaler(with_mean=False), num_col),
        ("bool", "passthrough", bool_col),
    ],
    sparse_threshold=1.0,
)

model = LocalOutlierFactor(
    n_neighbors=10,
    contamination=0.01,   # adjust as desired
    novelty=True,           # allow usage on new/unseen data
    n_jobs=-1
)

pipe = Pipeline([("prep", preprocess), ("clf", model)])

pipe.fit(X_train)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [354]:
scores = pipe["clf"].decision_function(pipe["prep"].transform(X_valid_feat))
raw_anom = -scores

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

y_valid["anom_score"] = anom_score

ap = average_precision_score(y_valid["target"], y_valid["anom_score"])
roc_auc = roc_auc_score(y_valid["target"], y_valid["anom_score"])

print(f"AP: {ap:.4f}")
print(f"AUC-ROC : {roc_auc:.4f}")

AP: 0.2649
AUC-ROC : 0.6469


/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_99129/3843939458.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_valid["anom_score"] = anom_score


In [331]:
submission = test_result(test_data, pipe)

submission.to_csv('Data/submission2.csv', index = False)

Extended Isolation Forest

In [279]:
from sklearn.ensemble import IsolationForest

class ExtendedIsolationForestDetector:
    """
    Wrapper for Extended Isolation Forest for use in scikit-learn pipelines.
    Extended Isolation Forest improves upon standard Isolation Forest by
    using hyperplanes at random angles instead of axis-parallel splits.
    """
    def __init__(self, contamination=0.1, random_state=42):
        self.contamination = contamination
        self.random_state = random_state
        self.detector = None
    
    def fit(self, X, y=None):
        self.detector = IsolationForest(
            contamination=self.contamination,
            random_state=self.random_state,
            n_estimators=1000
        )
        self.detector.fit(X)
        return self
    
    def predict(self, X):
        # Returns -1 for outliers, 1 for inliers (sklearn convention)
        return self.detector.predict(X)
    
    def decision_function(self, X):
        # Returns anomaly scores (more negative = more anomalous)
        return self.detector.score_samples(X)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_col),
        ("num", StandardScaler(with_mean=False), num_col),
        ("bool", "passthrough", bool_col),
    ],
    sparse_threshold=1.0,
)

eif = ExtendedIsolationForestDetector(
        contamination=0.1,
        random_state=42
    )

pipe = Pipeline([("prep", preprocess), ("clf", eif)])

pipe.fit(X_train)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [280]:
scores = pipe["clf"].decision_function(pipe["prep"].transform(X_valid))
raw_anom = -scores

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

y_valid["anom_score"] = anom_score

ap = average_precision_score(y_valid["target"], y_valid["anom_score"])
roc_auc = roc_auc_score(y_valid["target"], y_valid["anom_score"])

print(f"AP: {ap:.4f}")
print(f"AUC-ROC : {roc_auc:.4f}")

AP: 0.1301
AUC-ROC : 0.5345


/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_99129/2162249464.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_valid["anom_score"] = anom_score


Ensemble - LOF

In [281]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.utils import resample
from sklearn.neighbors import LocalOutlierFactor

class EnsembleLOF(BaseEstimator, TransformerMixin):
    def __init__(self, n_estimators=10, neighbor_list=None, sample_fraction=0.8, random_state=42):
        self.n_estimators = n_estimators
        self.neighbor_list = neighbor_list if neighbor_list is not None else [10, 20, 30, 50]
        self.sample_fraction = sample_fraction
        self.random_state = random_state
        self.models = []
    
    def _ensure_dense(self, X):
        from scipy.sparse import issparse
        if issparse(X):
            return X.toarray()
        return X

    def fit(self, X, y=None):
        X = self._ensure_dense(X)
        np.random.seed(self.random_state)
        self.models = []

        for i in range(self.n_estimators):
            # Bootstrap sampling
            X_sample = resample(
                X,
                replace=False,
                n_samples=int(len(X) * self.sample_fraction),
                random_state=self.random_state + i,
            )
            n_neighbors = np.random.choice(self.neighbor_list)
            lof = LocalOutlierFactor(
                n_neighbors=n_neighbors,
                contamination="auto",
                novelty=True
            )
            lof.fit(X_sample)
            self.models.append(lof)
        return self

    def transform(self, X):
        # Compute mean LOF scores from ensemble
        all_scores = []
        for lof in self.models:
            scores = lof.decision_function(X)  # LOF anomaly scores
            all_scores.append(scores)
        mean_scores = np.mean(np.vstack(all_scores), axis=0)
        return mean_scores.reshape(-1, 1)  # Shape: (n_samples, 1)

In [282]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_col),
        ("num", StandardScaler(with_mean=False), num_col),
        ("bool", "passthrough", bool_col),
    ],
    sparse_threshold=1.0,
)

elof = EnsembleLOF(
            n_estimators=10,
            neighbor_list=[10, 20, 30, 50, 100],
            sample_fraction=0.8,
            random_state=42
        )

pipe = Pipeline([("prep", preprocess), ("clf", elof)])

pipe.fit(X_train)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [285]:
scores = pipe["clf"].transform(pipe["prep"].transform(X_valid))
raw_anom = -scores

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

y_valid["anom_score"] = anom_score

ap = average_precision_score(y_valid["target"], y_valid["anom_score"])
roc_auc = roc_auc_score(y_valid["target"], y_valid["anom_score"])

print(f"AP: {ap:.4f}")
print(f"AUC-ROC : {roc_auc:.4f}")

AP: 0.2384
AUC-ROC : 0.6320


/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_99129/664101819.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_valid["anom_score"] = anom_score
